In [1]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

import sys

sys.path.append("../..")

In [2]:
from src.data.load_data import load_data
from src.models.kernel_regression import KernelRegression
from src.conformal_prediction.ustable_cp import UStableConformalPredictor

Load data

In [3]:
input_points, output_points = load_data("friedman1")

In [4]:
train_input_points, test_input_points, train_output_points, test_output_points = (
    train_test_split(input_points, output_points, random_state=0)
)

Instantiate predictor

In [5]:
# loss_name = "log_cosh"
# loss_params = {"alpha":1.}

loss_name = "pseudo_huber"
loss_params = {"alpha": 1.0}

# loss_name = "smoothed_pinball"
# loss_params = {"alpha":1., "tau":0.5}

In [6]:
predictor = KernelRegression(
    lam=0.5,
    kernel="laplacian",
    solver="lbfgs", loss_name=loss_name, loss_params=loss_params
)

Instantiate region predictor

In [7]:
conformal_predictor = UStableConformalPredictor(
    predictor, non_conformity_name="absolute"
)
region_predictor = conformal_predictor.fit_predict(
    train_input_points, train_output_points, test_input_points
)

In [8]:
confidence_control_level = 0.1
prediction_regions = region_predictor(confidence_control_level)

In [9]:
prediction_regions

[{'upper': [array([-1.68853629]),array([1.68824384])],
  'lower': [array([-1.6672597]),array([1.66696725])]},
 {'upper': [array([-1.6855965]),array([1.69117913])],
  'lower': [array([-1.6643199]),array([1.66990253])]},
 {'upper': [array([-1.68547752]),array([1.69129791])],
  'lower': [array([-1.66420092]),array([1.67002131])]},
 {'upper': [array([-1.68611982]),array([1.69065575])],
  'lower': [array([-1.66484323]),array([1.66937916])]},
 {'upper': [array([-1.68653332]),array([1.69024491])],
  'lower': [array([-1.66525672]),array([1.66896832])]},
 {'upper': [array([-1.68898516]),array([1.68779512])],
  'lower': [array([-1.66770856]),array([1.66651853])]},
 {'upper': [array([-1.68539703]),array([1.69137817])],
  'lower': [array([-1.66412043]),array([1.67010157])]},
 {'upper': [array([-1.68459825]),array([1.69217561])],
  'lower': [array([-1.66332166]),array([1.67089901])]},
 {'upper': [array([-1.68706684]),array([1.68971102])],
  'lower': [array([-1.66579024]),array([1.66843443])]},
 {'u

In [10]:
coverage_upper = np.mean(
    [
        test_output_point in prediction_region["upper"]
        for test_output_point, prediction_region in zip(
            test_output_points, prediction_regions
        )
    ]
)
print("test coverage: ", coverage_upper)

test coverage:  0.92


In [11]:
coverage_lower = np.mean(
    [
        test_output_point in prediction_region["lower"]
        for test_output_point, prediction_region in zip(
            test_output_points, prediction_regions
        )
    ]
)
print("test coverage: ", coverage_lower)

test coverage:  0.912
